# Step 2: Database Creation (DDL)

In [2]:
!pip install faker

In [9]:
# IMPORT LIBRARIES

import sqlite3
import pandas as pd
import random
from faker import Faker
fake = Faker()

In [4]:
# CREATE DATABASE

conn = sqlite3.connect("hajj_database.db")
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")


In [6]:
# Drop tables if exist (to avoid errors)
tables = ["Violations","Inspectors","Pilgrims","Contracts","Arrivals","Houses","Camps","Companies","Nationalities"]
for t in tables:
    cursor.execute(f"DROP TABLE IF EXISTS {t}")

In [7]:
# CREATE TABLES (DDL)

# Nationalities
cursor.execute("""
CREATE TABLE Nationalities (
    nationality_id INTEGER PRIMARY KEY,
    nationality_name_ar TEXT,
    nationality_name_en TEXT,
    country_name_ar TEXT,
    country_name_en TEXT,
    country_code TEXT,
    region TEXT
);
""")

# Companies
cursor.execute("""
CREATE TABLE Companies (
    company_id INTEGER PRIMARY KEY,
    company_name_ar TEXT,
    company_name_en TEXT,
    company_type TEXT,
    license_no TEXT,
    contact_person TEXT,
    contact_mobile TEXT,
    email TEXT,
    city TEXT,
    status TEXT,
    created_at DATETIME
);
""")

# Camps
cursor.execute("""
CREATE TABLE Camps (
    camp_id INTEGER PRIMARY KEY,
    camp_code TEXT,
    camp_name TEXT,
    mashair_location TEXT,
    latitude REAL,
    longitude REAL,
    area_sqm REAL,
    capacity INTEGER,
    allocated_quota INTEGER,
    camp_category TEXT,
    status TEXT
);
""")

# Houses
cursor.execute("""
CREATE TABLE Houses (
    house_id INTEGER PRIMARY KEY,
    house_name TEXT,
    city TEXT,
    district TEXT,
    house_type TEXT,
    classification TEXT,
    latitude REAL,
    longitude REAL,
    capacity INTEGER,
    available_rooms INTEGER,
    operator_name TEXT,
    status TEXT
);
""")

# Arrivals
cursor.execute("""
CREATE TABLE Arrivals (
    arrival_id INTEGER PRIMARY KEY,
    flight_no TEXT,
    airline_name TEXT,
    departure_country TEXT,
    departure_city TEXT,
    departure_airport TEXT,
    arrival_city TEXT,
    arrival_airport TEXT,
    scheduled_arrival_datetime DATETIME,
    actual_arrival_datetime DATETIME,
    delay_minutes INTEGER,
    terminal_no TEXT,
    gate_no TEXT,
    transport_type TEXT,
    arrival_status TEXT
);
""")

# Contracts
cursor.execute("""
CREATE TABLE Contracts (
    contract_id INTEGER PRIMARY KEY,
    contract_no TEXT,
    provider_type TEXT,
    camp_id INTEGER,
    house_id INTEGER,
    company_id INTEGER,
    contract_type TEXT,
    contract_start_date DATE,
    contract_end_date DATE,
    number_of_pilgrims INTEGER,
    contract_value_without_vat REAL,
    vat_rate REAL,
    vat_amount REAL,
    contract_value_with_vat REAL,
    payment_status TEXT,
    contract_status TEXT,
    created_at DATETIME,
    FOREIGN KEY (company_id) REFERENCES Companies(company_id),
    FOREIGN KEY (camp_id) REFERENCES Camps(camp_id),
    FOREIGN KEY (house_id) REFERENCES Houses(house_id)
);
""")

# Pilgrims
cursor.execute("""
CREATE TABLE Pilgrims (
    pilgrim_id INTEGER PRIMARY KEY,
    pilgrim_full_name TEXT,
    passport_no TEXT,
    visa_no TEXT,
    pilgrim_type TEXT,
    gender TEXT,
    age INTEGER,
    nationality_id INTEGER,
    company_id INTEGER,
    arrival_id INTEGER,
    mobile_no TEXT,
    emergency_contact TEXT,
    health_status TEXT,
    created_at DATETIME,
    FOREIGN KEY (nationality_id) REFERENCES Nationalities(nationality_id),
    FOREIGN KEY (company_id) REFERENCES Companies(company_id),
    FOREIGN KEY (arrival_id) REFERENCES Arrivals(arrival_id)
);
""")

# Inspectors
cursor.execute("""
CREATE TABLE Inspectors (
    visit_id INTEGER PRIMARY KEY,
    inspector_id INTEGER,
    inspector_name TEXT,
    inspector_team TEXT,
    inspector_mobile TEXT,
    company_id INTEGER,
    camp_id INTEGER,
    house_id INTEGER,
    visit_type TEXT,
    visit_datetime DATETIME,
    visit_status TEXT,
    overall_result TEXT,
    notes TEXT,
    FOREIGN KEY (company_id) REFERENCES Companies(company_id),
    FOREIGN KEY (camp_id) REFERENCES Camps(camp_id),
    FOREIGN KEY (house_id) REFERENCES Houses(house_id)
);
""")

# Violations
cursor.execute("""
CREATE TABLE Violations (
    violation_id INTEGER PRIMARY KEY,
    visit_id INTEGER,
    company_id INTEGER,
    camp_id INTEGER,
    house_id INTEGER,
    inspector_name TEXT,
    violation_category TEXT,
    checklist_question TEXT,
    answer TEXT,
    is_violation BOOLEAN,
    severity TEXT,
    violation_description TEXT,
    corrective_action TEXT,
    due_date DATE,
    closure_status TEXT,
    created_at DATETIME,
    FOREIGN KEY (visit_id) REFERENCES Inspectors(visit_id),
    FOREIGN KEY (company_id) REFERENCES Companies(company_id),
    FOREIGN KEY (camp_id) REFERENCES Camps(camp_id),
    FOREIGN KEY (house_id) REFERENCES Houses(house_id)
);
""")



# Step 3: Generated Data (Faker)

In [12]:

# Nationalities
nationalities = [{
    "nationality_id": i,
    "nationality_name_ar": fake.country(),
    "nationality_name_en": fake.country(),
    "country_name_ar": fake.country(),
    "country_name_en": fake.country(),
    "country_code": fake.country_code(),
    "region": fake.word()
} for i in range(1,21)]

# Companies
companies = [{
    "company_id": i,
    "company_name_ar": fake.company(),
    "company_name_en": fake.company(),
    "company_type": random.choice(["Hajj","Umrah"]),
    "license_no": fake.bothify("LIC####"),
    "contact_person": fake.name(),
    "contact_mobile": fake.phone_number(),
    "email": fake.email(),
    "city": random.choice(["Makkah","Madinah"]),
    "status": random.choice(["Active","Suspended"]),
    "created_at": fake.date_time_this_year()
} for i in range(1,16)]

# Camps
camps = [{
    "camp_id": i,
    "camp_code": f"C{i}",
    "camp_name": f"Camp {i}",
    "mashair_location": random.choice(["Mina","Arafat"]),
    "latitude": float(fake.latitude()),
    "longitude": float(fake.longitude()),
    "area_sqm": random.randint(1000,5000),
    "capacity": random.randint(100,500),
    "allocated_quota": random.randint(50,400),
    "camp_category": random.choice(["A","B"]),
    "status": "Active"
} for i in range(1,41)]

# Houses
houses = [{
    "house_id": i,
    "house_name": f"House {i}",
    "city": random.choice(["Makkah","Madinah"]),
    "district": fake.city(),
    "house_type": random.choice(["Hotel","Apartment"]),
    "classification": random.choice(["3-star","4-star"]),
    "latitude": float(fake.latitude()),
    "longitude": float(fake.longitude()),
    "capacity": random.randint(50,300),
    "available_rooms": random.randint(10,100),
    "operator_name": fake.company(),
    "status": "Active"
} for i in range(1,51)]

# Arrivals
arrivals = [{
    "arrival_id": i,
    "flight_no": fake.bothify("FL###"),
    "airline_name": fake.company(),
    "departure_country": fake.country(),
    "departure_city": fake.city(),
    "departure_airport": fake.word(),
    "arrival_city": "Jeddah",
    "arrival_airport": "KAIA",
    "scheduled_arrival_datetime": fake.date_time_this_year(),
    "actual_arrival_datetime": fake.date_time_this_year(),
    "delay_minutes": random.randint(0,120),
    "terminal_no": random.choice(["1","N"]),
    "gate_no": str(random.randint(1,50)),
    "transport_type": "Air",
    "arrival_status": random.choice(["On Time","Delayed"])
} for i in range(1,101)]

# Contracts
contracts = []
for i in range(1,101):
    contracts.append({
        "contract_id": i,
        "contract_no": f"CN{i}",
        "provider_type": random.choice(["Camp","House"]),
        "camp_id": random.choice(camps)["camp_id"] if random.random()>0.5 else None,
        "house_id": random.choice(houses)["house_id"] if random.random()>0.5 else None,
        "company_id": random.choice(companies)["company_id"],
        "contract_type": "Seasonal",
        "contract_start_date": fake.date_this_year(),
        "contract_end_date": fake.date_this_year(),
        "number_of_pilgrims": random.randint(50,500),
        "contract_value_without_vat": random.randint(10000,50000),
        "vat_rate": 0.15,
        "vat_amount": random.randint(1000,5000),
        "contract_value_with_vat": random.randint(20000,60000),
        "payment_status": random.choice(["Paid","Pending"]),
        "contract_status": random.choice(["Active","Expired"]),
        "created_at": fake.date_time_this_year()
    })

# Pilgrims
pilgrims = [{
    "pilgrim_id": i,
    "pilgrim_full_name": fake.name(),
    "passport_no": fake.bothify("??######"),
    "visa_no": fake.bothify("VISA####"),
    "pilgrim_type": random.choice(["Internal","External"]),
    "gender": random.choice(["Male","Female"]),
    "age": random.randint(18,80),
    "nationality_id": random.choice(nationalities)["nationality_id"],
    "company_id": random.choice(companies)["company_id"],
    "arrival_id": random.choice(arrivals)["arrival_id"],
    "mobile_no": fake.phone_number(),
    "emergency_contact": fake.phone_number(),
    "health_status": random.choice(["Good","Chronic"]),
    "created_at": fake.date_time_this_year()
} for i in range(1,5001)]

# Inspectors
inspectors = [{
    "visit_id": i,
    "inspector_id": i,
    "inspector_name": fake.name(),
    "inspector_team": "Team A",
    "inspector_mobile": fake.phone_number(),
    "company_id": random.choice(companies)["company_id"],
    "camp_id": random.choice(camps)["camp_id"],
    "house_id": random.choice(houses)["house_id"],
    "visit_type": "Inspection",
    "visit_datetime": fake.date_time_this_year(),
    "visit_status": "Completed",
    "overall_result": random.choice(["Pass","Fail"]),
    "notes": fake.text()
} for i in range(1,301)]

# Violations
violations = [{
    "violation_id": i,
    "visit_id": random.choice(inspectors)["visit_id"],
    "company_id": random.choice(companies)["company_id"],
    "camp_id": random.choice(camps)["camp_id"],
    "house_id": random.choice(houses)["house_id"],
    "inspector_name": fake.name(),
    "violation_category": "Safety",
    "checklist_question": fake.sentence(),
    "answer": random.choice(["Yes","No"]),
    "is_violation": random.choice([True,False]),
    "severity": random.choice(["Low","High"]),
    "violation_description": fake.text(),
    "corrective_action": fake.text(),
    "due_date": fake.date_this_year(),
    "closure_status": random.choice(["Open","Closed"]),
    "created_at": fake.date_time_this_year()
} for i in range(1,501)]
print(" Data genereted successfully")

 Data genereted successfully


In [13]:
# INSERT INTO DATABASE

pd.DataFrame(nationalities).to_sql("Nationalities", conn, if_exists="append", index=False)

pd.DataFrame(companies).to_sql("Companies", conn, if_exists="append", index=False)

pd.DataFrame(camps).to_sql("Camps", conn, if_exists="append", index=False)

pd.DataFrame(houses).to_sql("Houses", conn, if_exists="append", index=False)

pd.DataFrame(arrivals).to_sql("Arrivals", conn, if_exists="append", index=False)

pd.DataFrame(contracts).to_sql("Contracts", conn, if_exists="append", index=False)

pd.DataFrame(pilgrims).to_sql("Pilgrims", conn, if_exists="append", index=False)

pd.DataFrame(inspectors).to_sql("Inspectors", conn, if_exists="append", index=False)

pd.DataFrame(violations).to_sql("Violations", conn, if_exists="append", index=False)

conn.commit()

print(" Data inserted successfully")

 Data inserted successfully



# Step 5: SQL Queries


In [16]:
# Query 1: العلاقة غير المباشرة بين الحجاج والمخيمات أو المساكن من خلال الشركات والعقود
query = """
SELECT
    p.pilgrim_full_name,
    c.company_name_en,
    ct.contract_no,
    cp.camp_name,
    h.house_name

FROM Pilgrims p

JOIN Companies c
ON p.company_id = c.company_id

JOIN Contracts ct
ON c.company_id = ct.company_id

LEFT JOIN Camps cp
ON ct.camp_id = cp.camp_id

LEFT JOIN Houses h
ON ct.house_id = h.house_id

LIMIT 11;
"""

pd.read_sql_query(query, conn)

,pilgrim_full_name,company_name_en,contract_no,camp_name,house_name
0,Vanessa Rogers,"Ford, Smith and Colon",CN21,None,None
1,Vanessa Rogers,"Ford, Smith and Colon",CN36,Camp 9,House 20
2,Vanessa Rogers,"Ford, Smith and Colon",CN40,None,House 10
3,Vanessa Rogers,"Ford, Smith and Colon",CN49,Camp 15,House 44
4,Vanessa Rogers,"Ford, Smith and Colon",CN54,Camp 9,None
5,Vanessa Rogers,"Ford, Smith and Colon",CN55,Camp 40,None
6,Vanessa Rogers,"Ford, Smith and Colon",CN80,Camp 39,None
7,Vanessa Rogers,"Ford, Smith and Colon",CN89,Camp 14,None
8,Vanessa Rogers,"Ford, Smith and Colon",CN9,Camp 30,House 16
9,Beverly Evans,Taylor-Booth,CN100,Camp 32,House 7


In [18]:
# Query 2: المخالفات مع معلومات المفتش والشركة والمخيم والسكن.
query = """
SELECT

    v.violation_category,
    v.severity,

    i.inspector_name,

    c.company_name_en,

    cp.camp_name,

    h.house_name

FROM Violations v

JOIN Inspectors i
ON v.visit_id = i.visit_id

JOIN Companies c
ON v.company_id = c.company_id

LEFT JOIN Camps cp
ON v.camp_id = cp.camp_id

LEFT JOIN Houses h
ON v.house_id = h.house_id

LIMIT 11;
"""

pd.read_sql_query(query, conn)

,violation_category,severity,inspector_name,company_name_en,camp_name,house_name
0,Safety,High,Victor Johnson,Taylor-Booth,Camp 34,House 41
1,Safety,High,Jonathan Shelton,Jones Ltd,Camp 21,House 45
2,Safety,Low,Ashley Francis,Williams Group,Camp 39,House 48
3,Safety,Low,Nancy Melton,Bartlett-Hayes,Camp 20,House 7
4,Safety,Low,Kevin Patrick,Williams Group,Camp 6,House 7
5,Safety,High,Craig Jensen,Watts-Hernandez,Camp 32,House 1
6,Safety,Low,Brittany Harris,"Serrano, Tucker and Hall",Camp 14,House 32
7,Safety,Low,Samuel Morrison,Dunn-Allen,Camp 32,House 11
8,Safety,Low,Valerie Lawrence,"Ford, Smith and Colon",Camp 31,House 38
9,Safety,High,Taylor Crane,Williams Group,Camp 7,House 8


In [19]:
# Query 3: عدد المخالفات لكل شركة
query = """
SELECT company_id, COUNT(*) AS violations_count
FROM Violations
GROUP BY company_id;
"""
pd.read_sql_query(query, conn)

,company_id,violations_count
0,1,26
1,2,28
2,3,35
3,4,34
4,5,25
5,6,39
6,7,30
7,8,36
8,9,38
9,10,29


In [20]:
# Query 4: عدد الحجاج لكل شركة
conn = sqlite3.connect("hajj_database.db")

query = """
SELECT company_id, COUNT(*) AS total_pilgrims
FROM Pilgrims
GROUP BY company_id;
"""

df = pd.read_sql_query(query, conn)


In [21]:
# Query 5: عملية ربط داخلي (INNER JOIN) بين جدولي الحجاج والشركات
query= """
SELECT p.pilgrim_full_name, c.company_name_en
FROM Pilgrims p
JOIN Companies c ON p.company_id = c.company_id
LIMIT 10
"""
pd.read_sql_query(query, conn)

,pilgrim_full_name,company_name_en
0,Vanessa Rogers,"Ford, Smith and Colon"
1,Beverly Evans,Taylor-Booth
2,Jaime Gonzalez,Watts-Hernandez
3,James Booth,"Lara, Oconnor and Jones"
4,Carolyn Gomez,Rivera-Farley
5,Matthew Johnson,Jones Ltd
6,Lisa Donovan,Tucker-Lewis
7,Julie Diaz,Dunn-Allen
8,Janice Salazar,Bates-Gould
9,Debra Elliott,Silva Inc
